# Answers: 2 · Your first experiment

Worked answers to the **Try it yourself** exercises at the end of
[`2_your_first_experiment.ipynb`](../2_your_first_experiment.ipynb). Try each one yourself first. If you're stuck, run the cell here and
compare. Each answer runs on the same real data as the lesson, so you'll see real results,
not made-up ones.

The first code cell recreates just what the exercises need from the lesson, so this notebook
runs on its own.

The lesson's `run(...)` only took a backbone. Here it takes a few more options (`topk`,
the surrogate type), so all exercises can use one helper.

In [ ]:
import sys; sys.path.insert(0, "..")
import json, io, contextlib
import numpy as np
from tour import *
import run_experiment as rx
register_mini(rx)

def run(surrogate="single", backbones=("unimol2",), seed=42, acq="greedy", topk=100,
        init=210, batch=210, rounds=5):
    out = new_run_dir()
    which = ["--backbone", backbones[0]] if surrogate == "single" else ["--backbones", *backbones]
    sys.argv = ["run_experiment.py", "--dataset", "MINI", "--mode", "mve", "--surrogate", surrogate, *which,
                "--acq", acq, "--init-size", str(init), "--batch-size", str(batch), "--n-rounds", str(rounds),
                "--topk", str(topk), "--seed", str(seed), "--run-dir", str(out)]
    with contextlib.redirect_stdout(io.StringIO()):
        rx.main()
    return [h["topk_recall"] for h in json.loads((out / "history.json").read_text())]

SEEDS = [42, 43, 44]
print("ready")

---
## Exercise 1: which backbone finds the most?

One seed can be lucky, so we run 3 seeds per backbone.

In [ ]:
for bb in ["molformer", "unimol2", "mhgged"]:
    finals = [run(backbones=(bb,), seed=s)[-1] for s in SEEDS]
    print(f"{bb:10s} final recall {np.mean(finals):.2f}   seeds: {finals}")

**Answer.** Uni-Mol2 and MHG-GED tie at about 0.94. MoLFormer finds only 0.29, worse than a third as many, though still about 5× better than random (6%). Same loop, same budget, same model: only the embedding changed. Notebook 3 measured the reason directly. MoLFormer's embeddings order molecules by docking score much less well (Spearman 0.43, against 0.87–0.88 for the other two), so its "best guesses" are often not the best molecules. And notebook 4 shows what that weakness does to a combined model.

---
## Exercise 2: UCB instead of greedy

In [ ]:
for acq in ["greedy", "ucb"]:
    finals = [run(acq=acq, seed=s)[-1] for s in SEEDS]
    print(f"{acq:6s} final recall {np.mean(finals):.3f}   seeds: {finals}")

**Answer.** A small difference, in UCB's disfavour: 0.917 against greedy's 0.940, over 3 seeds. The seeds barely overlap (UCB 0.90–0.93, greedy 0.93–0.95), so it's probably a real, if small, cost. Each batch UCB spends partly on molecules the model is unsure about, and with a good model those are usually just average molecules. Notebook 4 explores this properly, and the project found the same on the full pool.

---
## Exercise 3: `--topk 1000` instead of 100

In [ ]:
for k in [100, 1000]:
    curve = run(topk=k)
    print(f"topk {k:4d}: recall per round {curve}")
print(f"\nrandom docking would find about {1260 / 21044:.0%} of either target")

**Answer.** Recall drops a lot: 0.55 instead of 0.94 after 5 rounds. Nothing about the search changed; the **target** got harder in two ways. First, the top 1,000 is 4.75% of the pool, and with 1,260 molecules docked in total, 210 of them random, you'd need to hit almost exclusively top-1,000 molecules to find them all. Second, the top 1,000 includes many "good, not great" molecules (down to about −10 kcal/mol) that look a lot like average ones, and those are much harder to tell apart than the extreme best 100. Random docking still finds only 6% of either target, so active learning is still about 9× better. **A recall number always depends on k**, which is why the project fixed k = 1,000 for its 2.1M pool, the same ratio as top-100 here (0.05%).